# GeoBrix Sample Data Setup

Run this notebook on a **Databricks cluster** to download the Essential and Complete sample-data bundles into a Unity Catalog Volume. Uses the GeoBrix **`sample`** module (included in the GeoBrix WHL)—install GeoBrix and run the cells in order.

- Skips datasets that already exist at the Volumes path
- Uses a local temp directory for interim downloads (zip, convert); only final files are copied to the Volume

## 1. Install GeoBrix (once per session)

The notebook installs the env6 wheel with the `stac` and `vizx` extras (they bring the STAC client and `geopandas` the downloaders use). It targets **Serverless environment 6** and classic **Databricks Runtime 19** clusters. On Serverless environment 5 or Databricks Runtime 17.3 / 18, install the `+env5` wheel instead. Point the path at your staged wheel if its Volume path differs.

In [0]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.6.0+env6-py3-none-any.whl"
%pip install --quiet "geobrix[stac,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.6.0+env6-py3-none-any.whl"

GeoBrix 0.6.0+env6 installed (build e523f16e6dfb, wheel sha256 70fac9419b5c) on Serverless environment 6


In [ ]:
%restart_python

## 2. Config: catalog, schema, volume

Set `CATALOG`, `SCHEMA` and `VOLUME` to a Unity Catalog Volume you can write to. The Volume must already exist. The bundles land under `/Volumes/<catalog>/<schema>/<volume>/geobrix-examples`.

In [0]:
from databricks.labs.gbx.sample import get_volumes_path, run_essential_bundle, run_complete_bundle
from pathlib import Path

# A Volume you can write to (it must already exist).
CATALOG = "main"
SCHEMA = "default"
VOLUME = "geobrix_samples"

In [0]:
SAMPLE_DATA_PATH = get_volumes_path(CATALOG, SCHEMA, VOLUME)
print(f"SAMPLE_DATA_PATH = {SAMPLE_DATA_PATH}")

SAMPLE_DATA_PATH = /Volumes/geospatial_docs/geobrix/data/geobrix-examples


In [0]:
## -- uncomment to remove everything under 'geobrix-examples' subfolder
#dbutils.fs.rm(SAMPLE_DATA_PATH, recurse=True)

## 3. Run Essential Bundle (~540 MB)

NYC & London vectors (GeoJSON), Sentinel-2 rasters, SRTM elevation. Skips files that already exist.

In [0]:
try:
    Path(SAMPLE_DATA_PATH).mkdir(parents=True, exist_ok=True)
except Exception:
    print("... VOLUME path must already exist.")
    pass
result_essential = run_essential_bundle(SAMPLE_DATA_PATH)
print(f"Files: {result_essential['file_count']}, Total: {result_essential['total_size_mb']:.1f} MB")
if result_essential["errors"]:
    for name, err in result_essential["errors"]:
        print(f"  ⚠️ {name}: {err[:80]}...")


📍 NYC Vector Data
⬇️  Downloading NYC Taxi Zones...
✅ NYC Taxi Zones: 3.7 MB
⬇️  Downloading NYC Boroughs...
✅ NYC Boroughs: 3.0 MB

📍 London Vector Data
⬇️  Downloading London Postcodes...
✅ London Postcodes: 0.9 MB

🛰️  NYC Raster Data
⬇️  Downloading Sentinel-2 NYC (scene: S2A_MSIL2A_20230601T155141_R011_T18TWL_20240812T201136)...
✅ Sentinel-2 NYC: 205.4 MB
✅ Sentinel-2 NYC (viewable): 115.0 MB

⛰️  NYC Elevation
⬇️  Downloading SRTM NYC...
✅ SRTM NYC: 0.0 MB

⛰️  London Elevation
⬇️  Downloading SRTM London...
✅ SRTM London: 0.0 MB

🛰️  London Raster Data
⬇️  Downloading Sentinel-2 London (scene: S2A_MSIL2A_20230707T105621_R094_T30UXB_20230707T194309)...
✅ Sentinel-2 London: 92.7 MB
✅ Sentinel-2 London (viewable): 115.0 MB
Files: 9, Total: 535.9 MB


## 4. Run Complete Bundle (~150 MB more)

Neighborhoods, extra elevation, HRRR weather, shapefiles (parks, subway), multi-layer GeoPackage, NYC File Geodatabase. Run after Essential.

In [0]:
result_complete = run_complete_bundle(SAMPLE_DATA_PATH)
print(f"Files: {result_complete['file_count']}, Total: {result_complete['total_size_mb']:.1f} MB")
if result_complete["errors"]:
    for name, err in result_complete["errors"]:
        print(f"  ⚠️ {name}: {err[:80]}...")


📍 Additional Geographic Data
⬇️  Downloading NYC Neighborhoods...
✅ NYC Neighborhoods: 4.1 MB
⬇️  Downloading SRTM NYC East...
✅ SRTM NYC East: 0.0 MB
⬇️  Downloading London Boroughs...
✅ London Boroughs: 1.7 MB

📦 Format Examples (shapefiles zipped; interim work in temp)
⬇️  Downloading HRRR Weather...
✅ HRRR Weather: 130.7 MB
⬇️  Downloading NYC Parks...
✅ NYC Parks: 2.1 MB
⬇️  Downloading NYC Subway...
✅ NYC Subway: 0.1 MB
⬇️  Building Multi-Layer GeoPackage...
✅ Multi-Layer GeoPackage: 7.0 MB
⬇️  Building NYC FileGDB...
  ✅ FileGDB NYC_Boroughs: 5 features
  ✅ FileGDB NYC_TaxiZones: 263 features
  ✅ FileGDB NYC_Parks: 500 features
  ✅ FileGDB NYC_SubwayStations: 2120 features
✅ NYC FileGDB: 1.0 MB
Files: 17, Total: 682.7 MB


## 5. Summary

In [0]:
print(f"Sample data base path: {SAMPLE_DATA_PATH}")
for item in sorted(Path(SAMPLE_DATA_PATH).rglob("*")):
    if item.is_file():
        rel = item.relative_to(SAMPLE_DATA_PATH)
        print(f"  {rel}: {item.stat().st_size / (1024*1024):.1f} MB")

Sample data base path: /Volumes/geospatial_docs/geobrix/data/geobrix-examples
  london/boroughs/london_boroughs.geojson: 1.7 MB
  london/elevation/srtm_n51w001.tif: 0.0 MB
  london/postcodes/london_postcodes.geojson: 0.9 MB
  london/sentinel2/london_sentinel2_red.tif: 92.7 MB
  london/sentinel2/london_sentinel2_red_byte.tif: 115.0 MB
  nyc/boroughs/nyc_boroughs.geojson: 3.0 MB
  nyc/elevation/srtm_n40w073.tif: 0.0 MB
  nyc/elevation/srtm_n40w074.tif: 0.0 MB
  nyc/filegdb/NYC_Sample.gdb.zip: 1.0 MB
  nyc/geopackage/nyc_complete.gpkg: 7.0 MB
  nyc/hrrr-weather/hrrr_nyc_20261111_12z.grib2: 130.7 MB
  nyc/neighborhoods/nyc_nta.geojson: 4.1 MB
  nyc/parks/nyc_parks.shp.zip: 2.1 MB
  nyc/sentinel2/nyc_sentinel2_red.tif: 205.4 MB
  nyc/sentinel2/nyc_sentinel2_red_byte.tif: 115.0 MB
  nyc/subway/nyc_subway.shp.zip: 0.1 MB
  nyc/taxi-zones/nyc_taxi_zones.geojson: 3.7 MB
